# 04 — Model Optimization and Evaluation

**Purpose.** Hyperparameter search for the LSTM, final comparison against baselines, confusion-matrix analysis, and error analysis.

**Selection metric:** validation macro-F1  
**Test set:** used once at the end for the best configuration


## 1. Setup


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.append(str(REPO_ROOT / 'src'))

import json
import itertools
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

from campaign_strategist.config import ARTIFACT_DIR, CAMPAIGN_CLASSES, PROCESSED_DATA_DIR, PROJECT_ROOT
from campaign_strategist.baselines import flatten_sequences
from campaign_strategist.model import save_model
from campaign_strategist.training import classification_metrics, predict_labels, train_lstm, transform_sequences
from campaign_strategist.viz import BLUE, ORANGE, apply_report_style, clean_label, despine, save_figure

apply_report_style()
RANDOM_STATE = 7
np.random.seed(RANDOM_STATE)

## 2. Load Data and Splits from Notebook 03


In [ ]:
x = np.load(PROCESSED_DATA_DIR / 'sequences_x.npy')
y = np.load(PROCESSED_DATA_DIR / 'labels_y.npy')
sample_index = pd.read_parquet(PROCESSED_DATA_DIR / 'sample_index.parquet')
scaler = joblib.load(ARTIFACT_DIR / 'sequence_scaler.joblib')

train_idx = np.load(ARTIFACT_DIR / 'train_idx.npy')
val_idx = np.load(ARTIFACT_DIR / 'val_idx.npy')
test_idx = np.load(ARTIFACT_DIR / 'test_idx.npy')

x_train = transform_sequences(x[train_idx], scaler)
x_val = transform_sequences(x[val_idx], scaler)
x_test = transform_sequences(x[test_idx], scaler)
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

print(f'train {len(train_idx):,} | val {len(val_idx):,} | test {len(test_idx):,}')

## 3. Hyperparameter Grid Search

We search a small grid over hidden size and learning rate. Sequence length stays fixed at 12 for this notebook because changing it would require rebuilding samples; that experiment can be added later if needed. Each run uses early stopping on validation loss; the ranking metric is **validation macro-F1**.


In [ ]:
grid = {
    'hidden_size': [64, 96, 128],
    'learning_rate': [5e-4, 8e-4, 1e-3],
}

rows = []
for hidden_size, learning_rate in itertools.product(grid['hidden_size'], grid['learning_rate']):
    model, history = train_lstm(
        x_train, y_train, x_val, y_val,
        hidden_size=hidden_size,
        learning_rate=learning_rate,
        epochs=20,
        batch_size=64,
        patience=4,
        random_state=RANDOM_STATE,
    )
    val_pred = predict_labels(model, x_val)
    metrics = classification_metrics(y_val, val_pred)
    rows.append({
        'hidden_size': hidden_size,
        'learning_rate': learning_rate,
        'epochs_run': int(history[-1]['epoch']),
        'best_val_loss': float(min(h['val_loss'] for h in history)),
        'val_accuracy': metrics['accuracy'],
        'val_macro_f1': metrics['macro_f1'],
    })
    print(rows[-1])

grid_results = pd.DataFrame(rows).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)
grid_results

## 4. Search Results


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
labels = [f"h={r.hidden_size}, lr={r.learning_rate:g}" for r in grid_results.itertuples()]
ax.barh(labels[::-1], grid_results['val_macro_f1'][::-1], color=BLUE)
ax.set_xlabel('Validation macro-F1')
ax.set_title('LSTM Hyperparameter Search')
despine(ax)
plt.tight_layout()
save_figure(fig, PROJECT_ROOT / 'reports' / 'figures' / 'optimization_macro_f1.png')

grid_results.to_csv(PROJECT_ROOT / 'reports' / 'tables' / 'optimization_results.csv', index=False)
best = grid_results.iloc[0].to_dict()
(ARTIFACT_DIR / 'best_hyperparameters.json').write_text(json.dumps(best, indent=2))
print('Best config:', best)
grid_results

## 5. Final Model: Retrain Best Configuration

Retrain on train+validation using the best hyperparameters, then evaluate **once** on the held-out test set.


In [ ]:
x_trainval = np.concatenate([x_train, x_val], axis=0)
y_trainval = np.concatenate([y_train, y_val], axis=0)

# Use a small slice of train as internal early-stopping monitor to avoid touching test.
rng = np.random.default_rng(RANDOM_STATE)
perm = rng.permutation(len(x_trainval))
cut = max(int(0.1 * len(perm)), 1)
inner_val = perm[:cut]
inner_train = perm[cut:]

final_model, final_history = train_lstm(
    x_trainval[inner_train], y_trainval[inner_train],
    x_trainval[inner_val], y_trainval[inner_val],
    hidden_size=int(best['hidden_size']),
    learning_rate=float(best['learning_rate']),
    epochs=25,
    batch_size=64,
    patience=5,
    random_state=RANDOM_STATE,
)

y_pred = predict_labels(final_model, x_test)
final_metrics = classification_metrics(y_test, y_pred)
print('Final test metrics:', {k: final_metrics[k] for k in ('accuracy', 'macro_f1')})

## 6. Confusion Matrix and Per-Class Metrics


In [ ]:
display_labels = [clean_label(c) for c in CAMPAIGN_CLASSES]
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=display_labels, xticks_rotation=45, cmap='Blues', ax=ax, colorbar=False
)
ax.set_title('Tuned LSTM Confusion Matrix (Test Set)')
plt.tight_layout()
save_figure(fig, PROJECT_ROOT / 'reports' / 'figures' / 'model_confusion_matrix.png')
print(classification_report(y_test, y_pred, target_names=CAMPAIGN_CLASSES, zero_division=0))

## 7. Error Analysis

We inspect the most confused class pairs and sample misclassified households. Errors between behaviorally similar styles (for example price-led coupon vs. loyalty reward) are more acceptable than errors between opposite strategies such as onboarding vs. win-back.


In [ ]:
cm = np.asarray(final_metrics['confusion_matrix'])
pairs = []
for i in range(len(CAMPAIGN_CLASSES)):
    for j in range(len(CAMPAIGN_CLASSES)):
        if i != j and cm[i, j] > 0:
            pairs.append((cm[i, j], CAMPAIGN_CLASSES[i], CAMPAIGN_CLASSES[j]))
pairs = sorted(pairs, reverse=True)[:8]
print('Top confused pairs (true -> predicted):')
for count, true_c, pred_c in pairs:
    print(f'  {count:4d}  {true_c} -> {pred_c}')

test_index = sample_index.iloc[test_idx].reset_index(drop=True).copy()
test_index['y_true'] = y_test
test_index['y_pred'] = y_pred
errors = test_index[test_index.y_true != test_index.y_pred].head(10)
errors.assign(
    true_label=lambda d: d.y_true.map(lambda i: CAMPAIGN_CLASSES[i]),
    pred_label=lambda d: d.y_pred.map(lambda i: CAMPAIGN_CLASSES[i]),
)[['household_id', 'end_week', 'true_label', 'pred_label']]

## 8. Final Comparison Table for the Report


In [ ]:
# Recompute baselines on the same held-out test set for a fair final table.
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

f_train = flatten_sequences(x_train)
f_test = flatten_sequences(x_test)
knn = KNeighborsClassifier(n_neighbors=15, weights='distance').fit(f_train, y_train)
rf = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_leaf=3,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=1,
).fit(f_train, y_train)

rows = [
    {'model': 'LSTM (tuned)', **{k: final_metrics[k] for k in ('accuracy', 'macro_f1')}},
    {'model': 'random_forest', **{k: classification_metrics(y_test, rf.predict(f_test))[k] for k in ('accuracy', 'macro_f1')}},
    {'model': 'knn', **{k: classification_metrics(y_test, knn.predict(f_test))[k] for k in ('accuracy', 'macro_f1')}},
]
final_table = pd.DataFrame(rows).sort_values('macro_f1', ascending=False).reset_index(drop=True)
final_table.to_csv(PROJECT_ROOT / 'reports' / 'tables' / 'model_comparison.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 4.5))
final_table.set_index('model')[['accuracy', 'macro_f1']].plot(kind='bar', ax=ax, color=[BLUE, ORANGE], rot=0)
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Final Model Comparison (Test Set)')
despine(ax)
plt.tight_layout()
save_figure(fig, PROJECT_ROOT / 'reports' / 'figures' / 'model_comparison.png')

save_model(final_model, str(ARTIFACT_DIR / 'journey_lstm.pt'))
(ARTIFACT_DIR / 'nb04_metrics.json').write_text(json.dumps({
    'best_hyperparameters': best,
    'final_test': {k: final_metrics[k] for k in ('accuracy', 'macro_f1')},
    'comparison': final_table.to_dict(orient='records'),
}, indent=2))
final_table

## 9. Conclusions

The tuned LSTM is evaluated against KNN and Random Forest on a household-held-out test set. Because labels are weak-supervision signals from future purchase behavior rather than true campaign response outcomes, strong metrics support the modeling approach but should not be over-interpreted as causal campaign lift. Limitations include the public sample size, retailer specificity, and rule-based labels. Future work could replace weak labels with approved campaign-response outcomes and test additional seasonal windows.

## References

Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation, 9*(8), 1735–1780.

Paszke, A., et al. (2019). PyTorch: An imperative style, high-performance deep learning library. *Advances in Neural Information Processing Systems, 32*.

Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.

Ratner, A., Bach, S. H., Ehrenberg, H., Fries, J., Wu, S., & Ré, C. (2017). Snorkel: Rapid training data creation with weak supervision. *Proceedings of the VLDB Endowment, 11*(3), 269–282.
